In [ ]:
import os
import subprocess
import textwrap
from pathlib import Path

import httpx
from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values
from IPython.display import Markdown

env_path = Path.cwd().parent / ".env.template"
for key, ref in dotenv_values(env_path).items():
    if ref is None:
        continue
    os.environ[key] = subprocess.run(
        ["op", "read", ref], capture_output=True, text=True, check=True
    ).stdout.strip()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

models = {
    "deepseek": "deepseek/deepseek-v4.1-flash",
    "llama": "meta-llama/llama-3.1-8b-instruct",
    "llama-70b": "meta-llama/llama-3.3-70b-instruct",
    "llama-flagship": "meta-llama/llama-4-maverick",
    "mistral": "mistralai/mistral-small-3.1-24b-instruct",
    "haiku": "anthropic/claude-haiku-4.5",
    "qwen": "qwen/qwen-2.5-coder-32b-instruct",
}
model = models["llama"]

In [90]:
def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def wrap_text(text: str, width: int = 120) -> str:
    return "\n".join(textwrap.fill(line, width=width) for line in text.splitlines())


def remaining_credits() -> float:
    response = httpx.get(
        "https://openrouter.ai/api/v1/credits",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=10.0,
    )
    response.raise_for_status()
    data = response.json()["data"]
    return data["total_credits"] - data["total_usage"]


# Start with an empty message list
messages: list[MessageParam] = []

In [76]:
def chat(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
) -> str:
    message = client.messages.create(
        model=model,
        max_tokens=10000,
        thinking={"type": "disabled"},
        messages=messages,
        system=system or omit,
        temperature=temperature,
        stop_sequences=stop_sequences or omit,
    )
    return get_reply(message)

In [32]:
add_user_message(messages, "Create a 1 sentence fake database description.")
with client.messages.stream(
    model=model,
    max_tokens=10000,
    thinking={"type": "disabled"},
    messages=messages,
    temperature=0.999,
) as stream:
    for text in stream.text_stream:
        print(wrap_text(f"{text}"), end="")

print("\n---")
print(wrap_text(get_reply(stream.get_final_message())))

The Chrono-Spatial Empathy Index is a fictional database that catalogs and cross-references the emotional resonance of historical events across parallel timelines.
---
The Chrono-Spatial Empathy Index is a fictional database that catalogs and cross-references the emotional resonance of
historical events across parallel timelines.


In [95]:
system = "You are a scientific assistant which popularizes complex scientific concepts."

while True:
    # You: Is it true that quantum computers consume hugh amount of energy? Why is it so?
    # You: how is physical qubit made? of what, silicone?
    user_input = input("You: ")
    if user_input == "/exit":
        break

    add_user_message(messages, user_input)
    print(wrap_text(f"You: {user_input}"))

    reply = chat(messages, system=system)
    add_assistant_message(messages, reply)

    print(wrap_text(f"AI: {reply}"))
    print("\n---\n")

You: Current size of observable Universe and it's size at the moment of time which we see now looking at the most
distant galaxies. not more than 3 sentences
AI: The observable universe is estimated to have a diameter of approximately 93 billion light-years, with a radius of
around 46.5 billion light-years. However, the age of the universe is about 13.8 billion years, so when we look at the
most distant galaxies, we see them as they existed around 13.5-13.6 billion years ago. This means that the observable
universe at that point in time was roughly 46-47 billion light-years in diameter.

---



In [94]:
model = models["llama"]

messages.clear()
add_user_message(messages, "Generate AWS EventBridge rule as JSON")
add_assistant_message(messages, "```json\n")
# Markdown(chat(messages, stop_sequences=["```"]).strip())
print(wrap_text(chat(messages, stop_sequences=["```"]).strip()))
print(f"Remaining: ${remaining_credits():.2f}")

{
  "Name": "my-rule",
  "EventPattern": "{\"source\":[\"aws.ec2\"],\"detail-type\":[\"EC2 Instance State-changed
Event\"],\"detail\":[{\"instance-id\":\"i-12345678\"}]}",
  "State": "ENABLED",
  "EventBusName": "default",
  "Targets": [
    {
      "Id": "my-target",
      "Arn": "arn:aws:logs:<region>:<account-id>:log-group:<log-group-name>",
      "RoleArn": "arn:aws:iam::<account-id>:role/<role-name>"
    }
  ],
  "InputTransformer": {
    "InputTemplates": [
      {
        "InputPathsMap": {
          "detail.instance-id": "$.detail.instance-id"
        },
        "InputTemplate": "{\"instance-id\": \"$.inputPathsMap.detail.instance-id}\""
      }
    ]
  },
  "RulePriority": 1
}
Remaining: $6.37


In [102]:
messages.clear()
add_user_message(messages, "Generate 3 different short AWS CLI commands")
add_assistant_message(
    messages, "Here are three different short AWS CLI commands without comments:\n```bash\n"
)
print(chat(messages, stop_sequences=["```"]).strip())
Markdown(f"---\nRemaining: **${remaining_credits():.2f}**")

aws ec2 start-instances --instance-ids i-123456
aws s3 ls
aws iam list-users


---
Remaining: **$6.37**